In [1]:
import Pkg; 
Pkg.activate(@__DIR__); 
Pkg.instantiate();

  Activating project at `~/Documents/optimal-control/code/with-notes/introduction-to-nonlinear-trajectory-optimization`
Precompiling project...
  ✓ JupyterFormatter
  ✓ SpecialFunctions
  ✓ DiffRules
  ✓ ColorVectorSpace → SpecialFunctionsExt
  ✓ ForwardDiff
  ✓ ForwardDiff → ForwardDiffStaticArraysExt
  ✓ ColorSchemes
  ✓ RobotDynamics
  ✓ RobotZoo
  ✓ PlotUtils
  ✓ PlotThemes
  ✓ RecipesPipeline
  ✓ Plots
  ✓ Plots → IJuliaExt
  ✓ Plots → UnitfulExt
  15 dependencies successfully precompiled in 58 seconds. 163 already precompiled.


In [2]:
using LaTeXStrings
using Plots
pyplot()
PyPlot.matplotlib.rc("text", usetex = true)
PyPlot.matplotlib.rc("font", family = "Times New Roman")
PyPlot.matplotlib.rc("text.latex", preamble = "\\usepackage{{amsmath}}")

using JupyterFormatter
enable_autoformat();

In [3]:
using LinearAlgebra
using ForwardDiff
using RobotZoo
using RobotDynamics;
# using MatrixCalculus

In [4]:
# acrobot dynamics
a = RobotZoo.Acrobot()
h = 0.05;

In [5]:
function dynamics_rk4(x, u)
    # RK4 integration with zero-order hold on u
    f1 = RobotDynamics.dynamics(a, x, u)
    f2 = RobotDynamics.dynamics(a, x + 0.5 * h * f1, u)
    f3 = RobotDynamics.dynamics(a, x + 0.5 * h * f2, u)
    f4 = RobotDynamics.dynamics(a, x + h * f3, u)
    return x + (h / 6.0) * (f1 + 2 * f2 + 2 * f3 + f4)
end;

In [6]:
function dfdx(x, u)
    ForwardDiff.jacobian(dx -> dynamics_rk4(dx, u), x)
end;

function dfdu(x, u)
    ForwardDiff.derivative(du -> dynamics_rk4(x, du), u)
end;

In [7]:
Nx = 4     # number of state
Nu = 1     # number of controls
Tfinal = 10.0 # final time
thist = Array(0:h:Tfinal)
Nt = length(thist);    # number of time steps

In [8]:
# Cost weights
Q = Diagonal([1.0 * ones(2); 0.1 * ones(2)]);
R = 0.01;
Qn = Array(100.0 * I(Nx));

In [9]:
function stage_cost(x, u)
    return 0.5 * ((x - xgoal)' * Q * (x - xgoal)) + 0.5 * R * u * u
end;

In [10]:
function terminal_cost(x)
    return 0.5 * (x - xgoal)' * Qn * (x - xgoal)
end;

In [11]:
function cost(xtraj, utraj)
    J = 0.0
    for k = 1:(Nt-1)
        J += stage_cost(xtraj[:, k], utraj[k])
    end
    J += terminal_cost(xtraj[:, Nt])
    return J
end;

In [12]:
#Initial guess
x0 = [-pi / 2; 0; 0; 0]
xgoal = [pi / 2; 0; 0; 0]
xtraj = kron(ones(1, Nt), x0)
utraj = randn(Nt - 1);

In [13]:
#Initial Rollout
for k = 1:(Nt-1)
    xtraj[:, k+1] .= dynamics_rk4(xtraj[:, k], utraj[k])
end;
J = cost(xtraj, utraj)

1493.8056835076864

\begin{equation}
    \begin{aligned}
        \min_{\substack{x_{1:N}\\u_{1:N-1}}}J=\sum_{k=1}^{N-1}{\frac{1}{2}x_{k}^{T}Q_kx_k+\frac{1}{2}u_{k}^{T}R_ku_k}+\underset{\text{LQR cost-to-go}}{{\frac{1}{2}x_{H}^{T}P_Hx_H}} \\
        \text{s.t.} \qquad x_{k+1} = A_k x_k + B_k u_k \\
        x_k\in \mathcal{X} \\
        u_k\in \mathcal{U} \\
        \mathcal{X} \text{ and } \mathcal{U} \text{ are convex.}
    \end{aligned}
\end{equation}

In [ ]:
# DDP / iLQR

    p = zeros(Nx, Nt)
    P = zeros(Nx, Nx, Nt)
    d = ones(Nt-1)
    K = zeros(Nu, Nx, Nt-1)
    ΔJ = 0.0
    
    xn = zeros(Nx, Nt)
    un = zeros(Nx, Nt-1)


gx = zeros(Nx)
gu = 0.0
Gxx = zeros(Nx, Nx)
Guu = 0.0
Gxu = zeros(Nx)
Gux = zeros(Nx)

iter = 0
while maximum(abs.(d)) > 1e-3
    iter += 1
    p = zeros(Nx, Nt)
    P = zeros(Nx, Nx, Nt)
    d = ones(Nt-1)
    K = zeros(Nu, Nx, Nt-1)
    ΔJ = 0.0

    p[:, end] = Qn*(xtraj[:, end]-xgoal)
    P[:, end] = Qn
    
    # backward pass
    for k = Nt-1:-1:1
        # calculate the derivatives
        q = Q*(xtraj[:,k]-xgoal)
        r = R*utraj[k]
        A=dfdx(xtraj[:, k], utraj[k])
        B = dfdu(xtraj[:, k], utraj[k])
        gx = q + A'*p[:, k+1]
        gu = r+B'p[:, k+1]
        Gxx = Q + A'*P[:,:,k+1]*A
        Guu = R + B'*P[:,:,k+1]*B
        Gxu = A'P[:,:,k+1]*B
        Gux = B'P[:,:,k+1]*A
        
        d[k]=Guu\gu
        K[:,:,k].=Guu\Gux
        
        p[:, k] .= gx-K[:,:,k]'*Gu+K[:,:,k]'*Guu*d[k]-Gxu*d[k]
        P[:,:,k].= Gxx + K[:,:,k]'*Guu*K[:,:,k] - Gxu*K[:,:,k] - K[:,:,k]'*Gux
    end
end;